# 00 · Setup & data catalog

**Purpose.** Verify the environment, load the typed research config, take a
census of every REAL data source this environment can see, and explain the
data architecture the other notebooks rely on.

**Contents**
1. [Environment & run manifest](#env)
2. [Research parameters](#params)
3. [Data architecture (raw → bronze → silver → gold)](#arch)
4. [Canonical identifiers](#ids)
5. [Real repo data sources census](#census)
6. [Raw-layer & dataset catalog (lineage)](#catalog)
7. [Connectivity probe](#conn)
8. [Findings, caveats, next steps](#findings)

**How to run.** Top-to-bottom, idempotent. `Params.offline = True` makes
every notebook read cached data only. Copy `config.example.yaml` →
`config.yaml` to change parameters persistently, or edit the parameter
cell below for a one-off.

<a id="env"></a>
## 1 · Environment & run manifest
Everything below is recorded to `outputs/run_logs/` so any result in this
folder can be traced to package versions + git commit.

In [1]:
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
JB = next(p for p in [_here, *_here.parents] if (p / "lib" / "bootstrap.py").exists())
sys.path.insert(0, str(JB))

import lib.bootstrap as bt
import pandas as pd

manifest = bt.run_manifest("00_setup_data_catalog")
pd.Series(manifest).to_frame("value")

,value
notebook,00_setup_data_catalog
run_utc,2026-07-04T13:26:32Z
git_commit,82618f4
python,3.9.6
pandas,2.3.3
polars,1.36.1
pyarrow,21.0.0
matplotlib,3.9.4
secrets_present,"{'ODDS_API_KEY': True, 'TELEGRAM_BOT_TOKEN': T..."


Credential **names** present (values are never read into notebook state):

In [2]:
pd.Series(bt.secret_names_present()).to_frame("configured")

,configured
ODDS_API_KEY,True
TELEGRAM_BOT_TOKEN,True
POLYMARKET_PRIVATE_KEY,True


<a id="params"></a>
## 2 · Research parameters
One typed dataclass drives every notebook. **What to change:** edit
`config.yaml` (persistent) or pass overrides in a notebook's parameter
cell (one-off). Unknown keys fail loudly.

In [3]:
import lib.config as cfg

p = cfg.load_params()          # defaults ← config.yaml (if present)
cfg.write_example_yaml()       # regenerate the example so docs never drift
p.to_frame()

,param,value,description,sensible_range
0,min_edge_raw,0.02,"Model prob minus market implied prob, pre-cost",0.00–0.10
1,min_edge_net,0.01,Edge after fees and expected slippage,0.00–0.08
2,pm_fee_rate,0.0,Flat PM fee rate (0 for most WC markets),0–0.02
3,pm_taker_fee_coeff,0.03,PM taker fee = coeff·p·(1−p) where charged,0–0.05
4,exchange_commission,0.02,GBP exchange commission on net winnings,0.00 (Smarkets promo)–0.06 (Betfair)
5,slippage_frac_of_spread,0.5,Expected fill = mid + this × half-spread,0 (fill at mid)–1 (fill at touch)
6,max_spread,0.06,Reject quotes with ask−bid above this (prob un...,0.02–0.15
7,min_depth_usd,50.0,Min $ executable within slippage cap,10–500
8,max_slippage,0.01,Max price move walking the book for our size,0.005–0.03
9,staleness_max_s,3600,Quote older than this is stale → reject,300–14400


<a id="arch"></a>
## 3 · Data architecture

| layer | dir | contents | guarantees |
|---|---|---|---|
| **raw** | `data/raw/<source>/<endpoint>/` | immutable gzip JSON payload + `.meta.json` (endpoint, redacted params, headers, status, retrieval UTC, sha256) | never modified; doubles as the offline cache |
| **bronze** | `data/bronze/` | source-shaped Parquet, minimal typing | 1:1 traceable to raw snapshots |
| **silver** | `data/silver/` | canonical events/markets/outcomes/quotes/trades keyed on stable IDs | source IDs always preserved |
| **gold** | `data/gold/` | features, fair values, opportunities, decisions, stress datasets | every build recorded in the lineage catalog |

All reads/writes go through `lib.storage`, so the lineage table (§6) is
complete by construction. Heavy transforms use **Polars** (lazy where the
data is big); conversion to **Pandas** happens only at display/plot
boundaries and always through `storage.to_pandas(df, reason=...)`.

<a id="ids"></a>
## 4 · Canonical identifiers
Defined in `lib/ids.py` (unit-tested):

* `competition_id` = `fifa-wc-2026`
* `event_id` = `wc2026:<home>__<away>__<YYYY-MM-DDTHH>Z` — canonical team
  names via production `wca.data.teamnames.canonical`, kickoff floored to
  the hour so venue feed skews collide to one ID.
* `market_id` = `<event_id>|<type>|<line>|<period>|<settlement>` —
  **settlement basis is part of the key**: 1X2 settles at 90 minutes, PM
  advancement includes ET+pens; they can never silently merge.
* `outcome_id` = `<market_id>|<outcome>`; `snapshot_id` = raw-layer path.

In [4]:
import datetime as dt
import lib.ids as ids
demo_ko = dt.datetime(2026, 7, 6, 19, 0, tzinfo=dt.timezone.utc)
demo_ev = ids.event_id("Korea Republic", "Czechia", demo_ko)   # alias-proof
{"event_id": demo_ev,
 "market_90min": ids.market_id(demo_ev, "1x2", settlement=ids.S_90MIN),
 "market_etpens": ids.market_id(demo_ev, "advance", settlement=ids.S_ETPENS)}

{'event_id': 'wc2026:south-korea__czech-republic__2026-07-06T19Z',
 'market_90min': 'wc2026:south-korea__czech-republic__2026-07-06T19Z|1x2||FT|90min',
 'market_etpens': 'wc2026:south-korea__czech-republic__2026-07-06T19Z|advance||FT|et-pens'}

<a id="census"></a>
## 5 · Real repo data sources census
Row counts computed NOW from the actual files — nothing asserted from
memory. The dev-box `wca.db` is a **stale ledger copy** (canonical ledger
lives on the Mac mini only, read over SSH when needed).

In [5]:
import json, sqlite3

def _rows(db, sql):
    try:
        with sqlite3.connect(f"file:{db}?mode=ro", uri=True) as c:
            c.execute("PRAGMA query_only=ON")
            return c.execute(sql).fetchone()
    except Exception as e:
        return (f"unavailable: {e}",)

census = []
def add(name, path, detail=""):
    path = pathlib.Path(path)
    census.append({
        "source": name, "path": str(path.relative_to(bt.REPO_ROOT)) if str(path).startswith(str(bt.REPO_ROOT)) else str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1e6, 1) if path.exists() else None,
        "detail": detail})

if bt.ORDERFLOW_DB.exists():
    n_tr, lo, hi = _rows(bt.ORDERFLOW_DB,
        "SELECT count(*), datetime(min(ts),'unixepoch'), datetime(max(ts),'unixepoch') FROM pm_trades")
    n_mk, = _rows(bt.ORDERFLOW_DB, "SELECT count(*) FROM pm_markets")
    add("PM orderflow (production capture)", bt.ORDERFLOW_DB,
        f"{n_tr:,} trades {lo} → {hi} UTC; {n_mk:,} markets")
else:
    add("PM orderflow (production capture)", bt.ORDERFLOW_DB)

if bt.DEV_WCA_DB.exists():
    n_snap, lo, hi = _rows(bt.DEV_WCA_DB,
        "SELECT count(*), min(ts_utc), max(ts_utc) FROM odds_snapshots")
    add("sportsbook odds snapshots (STALE dev copy)", bt.DEV_WCA_DB,
        f"{n_snap:,} rows {lo} → {hi}")
else:
    add("sportsbook odds snapshots (STALE dev copy)", bt.DEV_WCA_DB)

if bt.MODEL_PRED_LOG.exists():
    n_lines = sum(1 for _ in open(bt.MODEL_PRED_LOG))
    add("model 1X2 prediction log", bt.MODEL_PRED_LOG, f"{n_lines:,} rows (jsonl)")
else:
    add("model 1X2 prediction log", bt.MODEL_PRED_LOG)

for name, path, key in [
        ("match results (settled)", bt.RESULTS_JSON, "results"),
        ("advancement model-vs-PM feed", bt.ADVANCEMENT_JSON, "markets"),
        ("shipped bet recommendations", bt.BET_RECS_JSON, "recommendations"),
        ("promos catalog", bt.PROMOS_JSON, "promotions")]:
    if pathlib.Path(path).exists():
        d = json.loads(pathlib.Path(path).read_text())
        items = d.get(key) or d.get("recs") or []
        gen = (d.get("meta") or {}).get("generated") or d.get("generated") or "?"
        add(name, path, f"{len(items)} items; generated {gen}")
    else:
        add(name, path)

add("StatsBomb raw (WC 43/106 history)", bt.REPO_ROOT / "data" / "raw" / "statsbomb")
add("players.db (mini-only — expected absent here)", bt.REPO_ROOT / "data" / "players.db")
add("player_events.db (mini-only — expected absent here)", bt.REPO_ROOT / "data" / "player_events.db")

pd.DataFrame(census)

,source,path,exists,size_mb,detail
0,PM orderflow (production capture),data/pm_orderflow.db,True,1597.9,"2,086,434 trades 2025-06-09 15:37:11 → 2026-07..."
1,sportsbook odds snapshots (STALE dev copy),data/wca.db,True,723.2,"1,262,293 rows 2026-06-11T13:27:30.716212+00:0..."
2,model 1X2 prediction log,data/model_predictions_log.jsonl,True,0.5,"1,177 rows (jsonl)"
3,match results (settled),data/processed/wc2026_results.json,True,0.0,88 items; generated ?
4,advancement model-vs-PM feed,site/advancement_data.json,True,0.0,0 items; generated 2026-07-04 11:47:09 UTC
5,shipped bet recommendations,site/bet_recs.json,True,0.0,0 items; generated 2026-07-04 11:56:46 UTC
6,promos catalog,site/promos_data.json,True,0.0,0 items; generated 2026-07-04 10:36:25 UTC
7,StatsBomb raw (WC 43/106 history),data/raw/statsbomb,True,0.0,
8,players.db (mini-only — expected absent here),data/players.db,False,NaN,
9,player_events.db (mini-only — expected absent ...,data/player_events.db,False,NaN,


<a id="catalog"></a>
## 6 · Raw-layer inventory & dataset lineage catalog
Empty on first run — populated as notebooks 01–08 pull and build.

In [6]:
import lib.storage as st
raw_inv = st.list_raw()
print(f"raw snapshots: {raw_inv.height}")
raw_inv.tail(15).to_pandas()

raw snapshots: 44


,source,endpoint,snapshot,retrieved_utc,status,payload_kb
0,polymarket,clob_prices_history,20260704T132502Z__a813a12df1,2026-07-04T13:25:02Z,200,26.0
1,polymarket,clob_prices_history,20260704T132503Z__ed50217b49,2026-07-04T13:25:03Z,200,22.8
2,polymarket,clob_prices_history,20260704T132504Z__c9dbf4a843,2026-07-04T13:25:04Z,200,22.8
3,polymarket,dataapi_holders,20260704T132015Z__d2cb8bf508,2026-07-04T13:20:15Z,200,70.8
4,polymarket,dataapi_holders,20260704T132504Z__d2cb8bf508,2026-07-04T13:25:04Z,200,70.8
5,polymarket,gamma_events_wc,20260704T131953Z__ae3c5c0f3f,2026-07-04T13:19:53Z,200,10.8
6,polymarket,gamma_events_wc,20260704T132442Z__ae3c5c0f3f,2026-07-04T13:24:42Z,200,10.8
7,theoddsapi,/sports,20260704T131806Z__ee9e145f73,2026-07-04T13:18:06Z,200,25.0
8,theoddsapi,/sports/soccer_fifa_world_cup/events,20260704T131807Z__41fc6e4466,2026-07-04T13:18:07Z,200,1.5
9,theoddsapi,/sports/soccer_fifa_world_cup/events/9c7073ae2...,20260704T131811Z__96ee147c6b,2026-07-04T13:18:11Z,200,9.3


In [7]:
cat = st.catalog()
print(f"datasets (latest builds): {cat.height}")
cat.to_pandas()

datasets (latest builds): 23


,layer,dataset,path,rows,cols,schema_hash,built_utc,inputs,notebook,note,file_sha256
0,bronze,oddsapi_events,data/bronze/oddsapi_events.parquet,8,6,92a38186ce46,2026-07-04T13:25:45Z,"[""theoddsapi/sports_soccer_fifa_world_cup_even...",01,,e4a1eb5d638d03a57ff19d31128dc8cd47b8841cb80469...
1,bronze,oddsapi_odds,data/bronze/oddsapi_odds.parquet,602,12,7f69a67847c6,2026-07-04T13:25:45Z,"[""theoddsapi/sports_soccer_fifa_world_cup_odds...",01,"regions=uk markets=h2h,totals",40115f011fa866ff49f3f37a6f8128a667169219a4e609...
2,bronze,oddsapi_participants,data/bronze/oddsapi_participants.parquet,91,2,324368f7e906,2026-07-04T13:25:45Z,"[""theoddsapi/sports_soccer_fifa_world_cup_part...",01,,2bf29c94a17eb35ebf1a21654e31f9ba81fa3d97bffcb4...
3,bronze,oddsapi_scores,data/bronze/oddsapi_scores.parquet,14,9,87870345b359,2026-07-04T13:25:45Z,"[""theoddsapi/sports_soccer_fifa_world_cup_scor...",01,,b9c0abfb01ca9a7e881346302c2b818b9b1909b5c625d1...
4,bronze,pm_gamma_events,data/bronze/pm_gamma_events.parquet,2,45,c1777c472ea8,2026-07-04T13:24:42Z,"[""polymarket/gamma_events_wc/20260704T132442Z_...",02,,bd6f0c2b758a6d49d56ce0181aac4979b943ac02b66fe8...
5,bronze,pm_markets,data/bronze/pm_markets.parquet,1447,16,28d519c081f6,2026-07-04T13:25:46Z,"[""repo:data/pm_orderflow.db#pm_markets""]",02,,8a5ab652b35c6a3b956a9587dc5074bfc4a34da2a13da3...
6,bronze,pm_trades,data/bronze/pm_trades.parquet,2086434,12,486c6b41084e,2026-07-04T13:19:07Z,"[""repo:data/pm_orderflow.db#pm_trades""]",02,"full production orderflow capture, trade grain",a0fadad5d49daa0f8eee5cf2292c4c4503d2bfbd1b52ab...
7,gold,decisions_advancement_actionable,data/gold/decisions_advancement_actionable.par...,14,25,2ed65b16b64c,2026-07-04T13:26:04Z,"[""/Users/andrewdoherty/Desktop/Coding/World Cu...",06,,43c74909dfab65e7e0d619c79d49aa5f99cf8c407d11a1...
8,gold,decisions_props_withheld,data/gold/decisions_props_withheld.parquet,9,8,af6b5ef52303,2026-07-04T13:26:04Z,"[""/Users/andrewdoherty/Desktop/Coding/World Cu...",06,,b4f0c049962bfb44de39df4f377bd0998ab0de01759ec1...
9,gold,decisions_singles_actionable,data/gold/decisions_singles_actionable.parquet,3,26,5a389a99643b,2026-07-04T13:26:04Z,"[""/Users/andrewdoherty/Desktop/Coding/World Cu...",06,,a986044edced1d66628dbde6ee5dc6dd65009f41d8fc86...


<a id="conn"></a>
## 7 · Connectivity probe
Which live sources are reachable RIGHT NOW (Polymarket needs the VPN
route on this MacBook; the VPN severs LAN to the mini — known trade-off).
Skipped entirely when `offline=True`.

In [8]:
import requests
conn_rows = []
if p.offline:
    conn_rows.append({"target": "(all)", "status": "skipped — offline mode"})
else:
    for name, url in [
            ("polymarket gamma", "https://gamma-api.polymarket.com/events?limit=1"),
            ("polymarket clob", "https://clob.polymarket.com/ok"),
            ("polymarket data-api", "https://data-api.polymarket.com/trades?limit=1"),
            ("the-odds-api (unauth ping)", "https://api.the-odds-api.com/v4/sports?apiKey=invalid")]:
        try:
            r = requests.get(url, timeout=10)
            conn_rows.append({"target": name, "status": f"HTTP {r.status_code} (reachable)"})
        except Exception as e:
            conn_rows.append({"target": name, "status": f"UNREACHABLE: {type(e).__name__}"})
conn_df = pd.DataFrame(conn_rows)
conn_df

/Users/andrewdoherty/Desktop/Coding/World Cup Alpha/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,target,status
0,polymarket gamma,HTTP 200 (reachable)
1,polymarket clob,HTTP 200 (reachable)
2,polymarket data-api,HTTP 200 (reachable)
3,the-odds-api (unauth ping),HTTP 401 (reachable)


<a id="findings"></a>
## 8 · Findings, caveats, next steps
*(static text; the tables above are the evidence)*

**Findings.** The offline backbone is substantial and real: ~2M captured
PM trades with per-market metadata, 1.26M sportsbook odds snapshots
(2026-06-11→23 window), the full model 1X2 prediction log, settled
results, and a fresh advancement feed. That is enough to run notebooks
02–08 fully offline.

**Caveats.**
* Dev-box `wca.db` ledger tables are stale — bankroll numbers derived from
  it in notebook 06 are labelled accordingly.
* Sportsbook odds snapshots end 2026-06-23 on this machine — recent-match
  sportsbook convergence needs a live Odds API pull (notebook 01) or the
  mini's DB.
* Historical PM order books were never captured (books are live-only) —
  historical spread/depth are honestly `unavailable`, not reconstructed.

**Next.** Run notebook 01 (guarded live odds pull), then 02 (PM universe
build) — everything downstream reads their datasets from the catalog.

In [9]:
import lib.plotting as plot
plot.save_table(conn_df, "00_connectivity")
plot.save_table(pd.DataFrame(census), "00_source_census")
print("saved outputs/tables/00_connectivity.csv, 00_source_census.csv")

saved outputs/tables/00_connectivity.csv, 00_source_census.csv
